# Phase 4 Exclusion Checks

---

Ian Solberg 

March 2026

Professor Piao

---

 **execution only notebook** 
 
 `data_handling.py` contains data decisions and useful functions for simplifying model assignments*

In [1]:
import pandas as pd 
import numpy as np 
from data_handling import MrozHandler
import statsmodels.api as sm
import statsmodels.formula.api as smf
fp = 'data/Mroz.csv'
Mroz = MrozHandler(fp)

Full dataset shape
Number of rows: 753
Number of features: 22
Working subset shape:
Number of rows: 428
Number of features: 22


#### Contents:

- Naive OLS
- Robustness check (outlier trimming)
- City as a controlling variable
- IMR recalculation using only `mothed` or only `fatheduc` (which is a better predictor? doesn't pairing down parameters improve model?)


---

## Check 1: Naive OLS

---

In [2]:
Mroz.clear_caches()

naive = "lwage ~ exper + expersq + educ + kidslt6 + nwifeinc"

Mroz.set_dependent("lwage")
Mroz.add_independents("exper", "expersq")
Mroz.add_controls("educ", "kidslt6", "nwifeinc")

model = smf.ols(formula=naive, data=Mroz.working).fit(cov_type='HC1')
print(model.summary())

Caches cleared
Dependent variable set to: lwage
Independent variables: ['exper', 'expersq']
Control variables: ['educ', 'kidslt6', 'nwifeinc']
                            OLS Regression Results                            
Dep. Variable:                  lwage   R-squared:                       0.163
Model:                            OLS   Adj. R-squared:                  0.153
Method:                 Least Squares   F-statistic:                     17.37
Date:                Thu, 19 Mar 2026   Prob (F-statistic):           1.20e-15
Time:                        13:38:07   Log-Likelihood:                -429.95
No. Observations:                 428   AIC:                             871.9
Df Residuals:                     422   BIC:                             896.3
Df Model:                           5                                         
Covariance Type:                  HC1                                         
                 coef    std err          z      P>|z|      [0.025 

---

## Check 2: Outlier Trimming

---

In [3]:
def find_outliers(obj:MrozHandler, var:str):
    # Manual Tukey Fence implementation
    Q1 = obj.full[var].quantile(0.25)
    Q3 = obj.full[var].quantile(0.75)
    IQR = Q3 - Q1

    outliers = obj.full[(obj.full[var] < (Q1 - 1.5 * IQR)) | (obj.full[var] > (Q3 + 1.5 * IQR))]

    return outliers

# Set up IMR vector

Mroz.clear_caches() # clear api cache

import statsmodels.api as sm
from scipy.stats import norm

# calculate IMR (Probability vector of choosing to work) using Probit:

Mroz.set_dependent("work", full=True)
Mroz.add_independents("kidslt6", "nwifeinc", "educ", "agew", "motheduc", "fatheduc", full=True)
probit_result = sm.Probit(Mroz.get_y().astype(int), Mroz.get_X()).fit()
fitted = probit_result.fittedvalues
imr_values = norm.pdf(fitted) / norm.cdf(fitted)
imr_series = pd.Series(imr_values, index=Mroz.full.index)

print("To Populate probit result table:")
print(probit_result.summary())

Mroz.attach("IMR", imr_series, to_working=True) # attach result


# set up model with IMR:
Mroz.clear_caches()
Mroz.set_dependent("lwage", full=False)
Mroz.add_independents("exper", "expersq", full=False)
Mroz.add_controls("educ", "kidslt6", "nwifeinc", "IMR", full=False)


outliers = find_outliers(Mroz, "lwage")

working_trimmed = Mroz.working.copy().drop(outliers.index)

full_model_outlier_trimmed = smf.ols(formula=Mroz.get_formula(), data=working_trimmed).fit(cov_type='HC1')
print(full_model_outlier_trimmed.summary())

Caches cleared
Dependent variable set to: work
Independent variables: ['kidslt6', 'nwifeinc', 'educ', 'agew', 'motheduc', 'fatheduc']
Optimization terminated successfully.
         Current function value: 0.603822
         Iterations 5
To Populate probit result table:
                          Probit Regression Results                           
Dep. Variable:                   work   No. Observations:                  753
Model:                         Probit   Df Residuals:                      746
Method:                           MLE   Df Model:                            6
Date:                Thu, 19 Mar 2026   Pseudo R-squ.:                  0.1169
Time:                        13:38:07   Log-Likelihood:                -454.68
converged:                       True   LL-Null:                       -514.87
Covariance Type:            nonrobust   LLR p-value:                 1.349e-23
                 coef    std err          z      P>|z|      [0.025      0.975]
--------------------

---

## Check 3: City as a Controlling Variable 

---

In [4]:
city_formula = Mroz.get_formula() + " + city"
print(city_formula)

full_model_city = smf.ols(formula=city_formula, data=Mroz.working).fit(cov_type='HC1')
print(full_model_city.summary())

Mroz.clear_caches()

lwage ~ exper + expersq + educ + kidslt6 + nwifeinc + IMR + city
                            OLS Regression Results                            
Dep. Variable:                  lwage   R-squared:                       0.164
Model:                            OLS   Adj. R-squared:                  0.150
Method:                 Least Squares   F-statistic:                     12.51
Date:                Thu, 19 Mar 2026   Prob (F-statistic):           1.36e-14
Time:                        13:38:07   Log-Likelihood:                -429.88
No. Observations:                 428   AIC:                             875.8
Df Residuals:                     420   BIC:                             908.2
Df Model:                           7                                         
Covariance Type:                  HC1                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------

## Check 4: IMR Vector Recalculation with Mother/Father Education swaps

In [ ]:
def set_up_imr(*cols) -> MrozHandler:
    Mroz.clear_caches() # clear api cache every call

    import statsmodels.api as sm
    from scipy.stats import norm

    Mroz.set_dependent("work", full=True)
    Mroz.add_independents("kidslt6", *cols, full=True)
    probit_result = sm.Probit(Mroz.get_y().astype(int), Mroz.get_X()).fit()
    fitted = probit_result.fittedvalues
    imr_values = norm.pdf(fitted) / norm.cdf(fitted)
    imr_series = pd.Series(imr_values, index=Mroz.full.index)
    Mroz.attach("IMR", imr_series, to_working=True) 
    return Mroz

def run_full_model_with_imr(mroz_handler:MrozHandler) -> sm.regression.linear_model.RegressionResultsWrapper:
    mroz_handler.clear_caches()
    mroz_handler.set_dependent("lwage", full=False)
    mroz_handler.add_independents("exper", "expersq", full=False)
    mroz_handler.add_controls("educ", "kidslt6", "nwifeinc", "IMR", full=False)
    full_model= smf.ols(formula=mroz_handler.get_formula(), data=mroz_handler.working).fit(cov_type='HC1')
    print(full_model.summary())
    return full_model

print("STARTING CHECK 4: IMR Vector Recalculation with Mother/Father Education swaps")
print("=================================")
print("Running with mother only...")
print("=================================")

# with mother only
def set_up_imr_mother_only():
    return set_up_imr("nwifeinc", "educ", "agew", "motheduc")
MrozMotherOnlyIMR = set_up_imr_mother_only()

model1 = run_full_model_with_imr(MrozMotherOnlyIMR)
print("=================================")
print("Running with father only...")
print("=================================")
# with father only 
def set_up_imr_father_only():
    return set_up_imr("nwifeinc", "educ", "agew", "fatheduc")
MrozFatherOnlyIMR = set_up_imr_father_only()

model2 = run_full_model_with_imr(MrozFatherOnlyIMR)
print("=================================")
print("Running with neither...")
print("=================================")
# with neither
def set_up_imr_neither():
    return set_up_imr("nwifeinc", "educ", "agew")
MrozNeitherIMR = set_up_imr_neither()

model3 = run_full_model_with_imr(MrozNeitherIMR)

STARTING CHECK 4: IMR Vector Recalculation with Mother/Father Education swaps
Running with mother only...
Caches cleared
Dependent variable set to: work
Independent variables: ['kidslt6', 'nwifeinc', 'educ', 'agew', 'motheduc']
Optimization terminated successfully.
         Current function value: 0.603947
         Iterations 5
Attached 'IMR' to dataset
Caches cleared
Dependent variable set to: lwage
Independent variables: ['exper', 'expersq']
Control variables: ['educ', 'kidslt6', 'nwifeinc', 'IMR']
                            OLS Regression Results                            
Dep. Variable:                  lwage   R-squared:                       0.163
Model:                            OLS   Adj. R-squared:                  0.151
Method:                 Least Squares   F-statistic:                     13.69
Date:                Thu, 19 Mar 2026   Prob (F-statistic):           3.16e-14
Time:                        13:38:07   Log-Likelihood:                -429.95
No. Observations:   